In [ ]:
import os
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model
from trl import SFTTrainer
from huggingface_hub import login
from transformers import AutoConfig

In [ ]:
MODEL_ID = "microsoft/Phi-3-mini-4k-instruct"
DATASET_ID = "keivalya/MedQuad-MedicalQnADataset"
OUTPUT_DIR = "./results_phi3_medical"

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

In [ ]:
os.environ["WANDB_DISABLED"] = "true"

In [ ]:
config=AutoConfig.from_pretrained(MODEL_ID)
config.rope_scaling = None

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    config = config
    device_map="auto",
    dtype=torch.float16,
    trust_remote_code=True
)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
model = prepare_model_for_kbit_training(model)

In [ ]:
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["qkv_proj", "o_proj", "gate_up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

In [ ]:
dataset = load_dataset(DATASET_ID, split="train")

def format_prompts(batch):
    formatted_texts = []
    for input_text, output_text in zip(batch['Question'], batch['Answer']):
        # Qwen ChatML syntax
        text = (
            f"<|im_start|>system\nYou are a helpful medical assistant.<|im_end|>\n"
            f"<|im_start|>user\n{input_text}<|im_end|>\n"
            f"<|im_start|>assistant\n{output_text}<|im_end|>"
        )
        tokens = tokenizer.encode(text, max_length=512, truncation=True)
        truncated_text = tokenizer.decode(tokens)

        formatted_texts.append(truncated_text)

    return {"text": formatted_texts}

print("Formatting dataset rows...")
dataset = dataset.map(format_prompts, batched=True)
dataset_split = dataset.train_test_split(test_size=0.1)

In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    optim="paged_adamw_8bit",
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    num_train_epochs=1,


    bf16=False,
    fp16=False,
    report_to="none",
    logging_strategy="steps",
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
)

In [ ]:
def trainer_formatting_func(example):
    return example["text"]

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset_split["train"],
    eval_dataset=dataset_split["test"],
    peft_config=peft_config,
    processing_class=tokenizer,
    formatting_func=trainer_formatting_func,
    args=training_args,
)

In [ ]:
trainer.train()